# Qwen GDN key Gram matrices

For one text, this renders an interactive per-head $KK^\top$ viewer for every Gated DeltaNet layer. `interp-engine` captures the exact L2-normalized keys consumed by the GDN recurrence, so each head's diagonal is approximately one.

In [ ]:
import math
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from interp_engine import Address, load_model, run_with_cache

MODEL_ID = 'Qwen/Qwen3.5-9B'
TEXT = 'The early morning rain turned the empty street into a mirror of blue city lights.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'float16' if device == 'cuda' else 'float32'
model = load_model(MODEL_ID, backend='eager', device=device, dtype=dtype)


In [ ]:
gdn_layers = [
    (name, module) for name, module in model.hf_model.named_modules()
    if 'gateddeltanet' in type(module).__name__.lower()
]
if not gdn_layers:
    raise RuntimeError(f'No GDN layers found in {MODEL_ID}')

input_ids = model.to_tokens(TEXT)
addresses = [Address('gdn_k', layer.layer_idx) for _, layer in gdn_layers]
cache = run_with_cache(model, input_ids, addresses)

kk_t = {}
for address in addresses:
    keys = cache[address][0].float().cpu()  # [token, value head, key dimension]
    kk_t[address.layer] = torch.einsum('thd,shd->hts', keys, keys)
    print(f'layer {address.layer:2}: mean per-head ‖K‖ = {keys.norm(dim=-1).mean():.6f}')

tokens = model.to_str_tokens(input_ids[0])
print(f'Captured {len(kk_t)} GDN layers for {len(tokens)} tokens.')


In [ ]:
layer_ids = sorted(kk_t)
head_count = next(iter(kk_t.values())).size(0)
if any(matrices.size(0) != head_count for matrices in kk_t.values()):
    raise ValueError('GDN layers have different head counts; this viewer expects one shared grid.')

columns = min(4, head_count)
rows = math.ceil(head_count / columns)
fig = make_subplots(
    rows=rows, cols=columns,
    subplot_titles=[f'Head {head}' for head in range(head_count)],
)
for layer_index, layer_id in enumerate(layer_ids):
    for head in range(head_count):
        row, column = divmod(head, columns)
        matrix = kk_t[layer_id][head].masked_fill(
            torch.triu(torch.ones_like(kk_t[layer_id][head], dtype=torch.bool), diagonal=1), float('nan')
        )
        fig.add_trace(go.Heatmap(
            z=matrix.numpy(), x=tokens, y=tokens, colorscale='Blues',
            zmin=0, zmax=1, visible=layer_index == 0,
            showscale=head == head_count - 1,
            colorbar={'title': 'KKᵀ'} if head == head_count - 1 else None,
            hovertemplate='query %{y}<br>key %{x}<br>KKᵀ: %{z:.4f}<extra></extra>',
        ), row=row + 1, col=column + 1)

fig.update_xaxes(tickangle=45, tickfont_size=9)
fig.update_yaxes(autorange='reversed', tickfont_size=9)
fig.update_layout(
    title=f'GDN layer {layer_ids[0]} — per-head key Gram matrices',
    width=360 * columns, height=340 * rows,
    updatemenus=[{
        'buttons': [{
            'label': f'GDN layer {layer_id}',
            'method': 'update',
            'args': [
                {'visible': [index // head_count == layer_index for index in range(len(fig.data))]},
                {'title': f'GDN layer {layer_id} — per-head key Gram matrices'},
            ],
        } for layer_index, layer_id in enumerate(layer_ids)],
        'direction': 'down', 'x': 0, 'xanchor': 'left', 'y': 1.12, 'yanchor': 'top',
    }],
)

HTML_PATH = 'gdn_kk_t.html'
fig.write_html(HTML_PATH, include_plotlyjs=True)
fig.show()
print(f'Interactive viewer saved to {HTML_PATH}')
